<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/elliptic-pde/dtb_elliptic_single_mode_algorithm_diagnostics_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DTB single-mode algorithm diagnostics

This notebook diagnoses the $d=10$ single-mode Poisson experiment by changing one numerical ingredient at a time: tangent dimension $R$, quadrature size $N$, parameter-selection strategy, and outer-loop evolution.

In [ ]:
# Clone the requested branch in a fresh Colab session.
import os, subprocess
from pathlib import Path
if Path('DTB_elliptic_utils.py').exists():
    ROOT = Path.cwd()
else:
    ROOT = Path('/content/dtb-colab-experiments')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', '-b', 'elliptic-pde', 'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(ROOT)], check=True)
    os.chdir(ROOT)
print('repository:', ROOT)

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import torch
from DTB_elliptic_utils import MLP, flatten_parameters
from dtb_algorithm_diagnosis import (
    SingleModeConfig,
    print_compact_summary,
    run_configuration_sweep,
    run_single_mode_configuration,
    select_tangent_indices,
    selection_counts,
)

torch.set_default_dtype(torch.float64)
DEVICE = torch.device('cpu')

## Parameter guide

- **$d$ — spatial dimension.** The domain is $[-1,1]^d$. This notebook uses $d=10$, where the exact mode is strongly concentrated away from most uniformly sampled points.
- **$p$ — total neural-network parameter count.** It is determined by $d$, width, and depth. DTB does not use all $p$ tangent coordinates here.
- **$R$ — tangent dimension.** DTB selects $R$ parameter derivatives $J_i=\partial_{\theta_i}T_\theta$. Larger $R$ gives a richer trial space, but also enlarges and can destabilize the $R\times R$ stiffness matrix. Always report the selection strategy with $R$.
- **$N$ — training quadrature size.** These $N$ spatial points assemble $G_\theta$ and $b_\theta$. Increasing $R$ without increasing $N$ can fit quadrature error. There is no universal safe ratio, so this notebook reports $N/R$, conditioning, validation error, and energy gaps.
- **$N_{\rm val}$ — independent validation size.** It is never used to construct the inner Ritz system. It measures generalization beyond the training quadrature.
- **Selection strategy.** 'random' samples from the flattened parameter vector. 'layer_balanced' allocates nearly equal counts per layer and retains one bias coordinate per layer, preventing large early-layer blocks from dominating in high dimension.
- **Width and depth.** These control the nonlinear network whose selected parameter derivatives form the tangent basis.
- **$K$ — outer steps.** Each step re-solves the inner coefficients $\alpha$, computes the fixed-$\alpha$ envelope direction, and updates $\theta$.
- **ridge_relative.** The inner solve uses $(G+\lambda I)\alpha=b$ with $\lambda$ scaled by the mean diagonal of $G$. It stabilizes small eigenvalues but changes the inner problem.
- **Probe points.** A fixed independent set used only to compare tangent features and tangent subspaces across outer iterations.

Use $R$ to study trial-space capacity and $N$ to study integration accuracy. Change only one of them within each sweep.

In [ ]:
# Central experiment controls. Reduce the largest values for a quick smoke test.
D, WIDTH, DEPTH = 10, 16, 2
VALIDATION_N, PROBE_N = 4096, 256
RIDGE_RELATIVE = 1.0e-4

R_VALUES = [16, 32, 64, 128]
FIXED_N_FOR_R_SWEEP = 768

N_VALUES = [768, 2048, 4096, 16384]
FIXED_R_FOR_N_SWEEP = 32
SAMPLE_SWEEP_SELECTION = 'layer_balanced'

OUTER_R, OUTER_N, OUTER_STEPS = 32, 4096, 12
STRATEGIES = ['random', 'layer_balanced']

def configuration(**changes):
    values = dict(
        dimension=D, width=WIDTH, depth=DEPTH,
        validation_count=VALIDATION_N, probe_count=PROBE_N,
        ridge_relative=RIDGE_RELATIVE,
        model_seed=100 + D, quadrature_seed=10 + D,
        selection_seed=700 + D,
    )
    values.update(changes)
    return SingleModeConfig(**values)

## Diagnostic 1 — selected-coordinate allocation

**Purpose.** Check whether the same nominal $R$ produces comparable layer coverage. A flattened-random rule can select mostly early-layer weights because that block grows with $d$. The balanced rule should cover every layer, including the output layer.

In [ ]:
torch.manual_seed(100 + D)
audit_model = MLP(D, width=WIDTH, depth=DEPTH)
audit_theta, audit_spec = flatten_parameters(audit_model)
allocation = {}
for strategy in STRATEGIES:
    indices = select_tangent_indices(
        audit_theta.numel(), audit_spec, FIXED_R_FOR_N_SWEEP,
        strategy, 700 + D,
    )
    allocation[strategy] = selection_counts(indices, audit_spec)

layers = list(next(iter(allocation.values())).keys())
x = np.arange(len(layers)); width = 0.36
fig, axis = plt.subplots(figsize=(8, 4))
for offset, strategy in enumerate(STRATEGIES):
    axis.bar(x + (offset - .5) * width, [allocation[strategy][layer] for layer in layers], width, label=strategy)
axis.set(xticks=x, xticklabels=layers, ylabel='selected coordinates', title=f'Layer allocation at R={FIXED_R_FOR_N_SWEEP}')
axis.grid(axis='y', alpha=.25); axis.legend(); plt.tight_layout(); plt.show()
print('total parameters p =', audit_theta.numel())

## Diagnostic 2 — increase $R$ while holding $N$ fixed

**Purpose.** Test whether extra tangent capacity improves the PDE approximation or merely fits the fixed quadrature. Interpret the metrics together:

- validation relative $L^2$: lower is better;
- $u_{\rm DTB}(0)$: should approach the exact value $1$;
- $\kappa(G)$: rapid growth signals near-dependent tangent features;
- $F_{\rm val}-F_{\rm train}$: a growing positive gap indicates empirical-quadrature overfitting.

Here $N$ remains exactly fixed, so $N/R$ decreases as $R$ grows.

In [ ]:
r_configurations = [
    configuration(
        train_count=FIXED_N_FOR_R_SWEEP,
        tangent_dimension=R,
        selection=strategy,
        outer_steps=0,
    )
    for strategy in STRATEGIES for R in R_VALUES
]
r_results = run_configuration_sweep(r_configurations, device=DEVICE)
for result in r_results:
    print_compact_summary(result)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for strategy in STRATEGIES:
    selected = [result for result in r_results if result['config']['selection'] == strategy]
    R = [result['config']['tangent_dimension'] for result in selected]
    summary = [result['summary'] for result in selected]
    axes[0, 0].plot(R, [row['relative_l2'] for row in summary], 'o-', label=strategy)
    axes[0, 1].plot(R, [row['center_value'] for row in summary], 'o-', label=strategy)
    axes[1, 0].semilogy(R, [row['condition_number'] for row in summary], 'o-', label=strategy)
    axes[1, 1].plot(R, [row['validation_F'] - row['train_F'] for row in summary], 'o-', label=strategy)
axes[0, 1].axhline(1.0, color='k', ls='--', lw=1, label='exact')
titles = ['validation relative L2', 'center value u(0)', 'condition number of G', 'validation F - training F']
for axis, title in zip(axes.flat, titles):
    axis.set(xlabel='tangent dimension R', title=title); axis.grid(alpha=.25); axis.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Diagnostic 3 — increase $N$ while holding $R$ and the selected basis fixed

**Purpose.** Isolate spatial-integration error. All runs reuse the same initialized network, selection seed, $R$, and selection strategy. Only the prefix length of the same scrambled Sobol sequence changes.

- relative $L^2$ and $u(0)$ measure solution improvement;
- exact-energy quadrature error compares the empirical energy of the known exact solution with $-d\pi^2/8$;
- the effective sample size of $u_*^2$ shows how few samples carry most solution mass;
- the central count records points satisfying $\|x\|_\infty<0.5$.

In [ ]:
n_configurations = [
    configuration(
        train_count=N,
        tangent_dimension=FIXED_R_FOR_N_SWEEP,
        selection=SAMPLE_SWEEP_SELECTION,
        outer_steps=0,
    )
    for N in N_VALUES
]
n_results = run_configuration_sweep(n_configurations, device=DEVICE)
for result in n_results:
    print_compact_summary(result)

In [ ]:
N = [result['config']['train_count'] for result in n_results]
summary = [result['summary'] for result in n_results]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0, 0].semilogx(N, [row['relative_l2'] for row in summary], 'o-')
axes[0, 1].semilogx(N, [row['center_value'] for row in summary], 'o-')
axes[0, 1].axhline(1.0, color='k', ls='--', lw=1, label='exact')
energy_error = [abs(row['exact_train_F'] - row['exact_continuous_F']) / abs(row['exact_continuous_F']) for row in summary]
axes[1, 0].loglog(N, energy_error, 'o-')
axes[1, 1].semilogx(N, [row['u_squared_effective_sample_size'] for row in summary], 'o-', label='ESS of u*²')
axes[1, 1].semilogx(N, [row['central_sample_count'] for row in summary], 's--', label='central points')
titles = ['validation relative L2', 'center value u(0)', 'relative exact-energy quadrature error', 'where useful samples occur']
for axis, title in zip(axes.flat, titles):
    axis.set(xlabel='training quadrature size N', title=title); axis.grid(alpha=.25)
axes[0, 1].legend(); axes[1, 1].legend(); plt.tight_layout(); plt.show()

## Diagnostic 4 — does the outer loop update the tangent basis?

**Purpose.** Separate optimizer activity from a meaningful trial-space change.

- relative_theta_change: cumulative parameter movement;
- relative_feature_change: raw Frobenius change of $J_\theta$ on fixed probe points;
- maximum_subspace_sine: sine of the largest principal angle between the initial and current feature spans; it remains near zero if features only rescale or mix inside the same span;
- relative_prediction_change: total change in the re-solved DTB prediction on probe points;
- accepted step and gradient norm: whether Armijo permits meaningful motion;
- training $F$ versus validation $L^2$: whether the changed basis actually improves the PDE approximation.

A decreasing training energy with flat subspace rotation or worsening validation error means that the outer loop is not providing useful basis adaptation.

In [ ]:
outer_configurations = [
    configuration(
        train_count=OUTER_N,
        tangent_dimension=OUTER_R,
        selection=strategy,
        outer_steps=OUTER_STEPS,
    )
    for strategy in STRATEGIES
]
outer_results = run_configuration_sweep(outer_configurations, device=DEVICE)
for result in outer_results:
    print_compact_summary(result)

In [ ]:
metrics = [
    ('train_F', 'training F', False),
    ('relative_l2', 'validation relative L2', False),
    ('relative_theta_change', 'relative theta change', False),
    ('relative_prediction_change', 'relative prediction change', False),
    ('accepted_step', 'accepted Armijo step', True),
    ('gradient_norm', 'outer gradient norm', True),
    ('relative_feature_change', 'raw tangent-feature change', False),
    ('maximum_subspace_sine', 'maximum tangent-subspace sine', False),
]
fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for result in outer_results:
    strategy = result['config']['selection']; history = result['history']
    steps = np.array([row['outer_step'] for row in history])
    for axis, (key, title, logarithmic) in zip(axes.flat, metrics):
        values = np.array([row[key] for row in history], dtype=float)
        valid = np.isfinite(values)
        if logarithmic:
            axis.semilogy(steps[valid], values[valid], 'o-', label=strategy)
        else:
            axis.plot(steps[valid], values[valid], 'o-', label=strategy)
        axis.set(xlabel='outer iteration', title=title)
for axis in axes.flat:
    axis.grid(alpha=.25); axis.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Reading the diagnostics

Evidence of **quadrature limitation**: accuracy improves strongly with $N$, the exact-energy quadrature error falls, and useful-sample ESS remains much smaller than $N$.

Evidence of **tangent overfitting**: increasing $R$ lowers training $F$ while validation error, energy gap, or $\kappa(G)$ grows.

Evidence of **selection bias**: flattened-random allocation misses later layers and performs inconsistently relative to layer-balanced selection.

Evidence of an **ineffective outer loop**: tiny accepted steps, little subspace rotation, and decreasing training $F$ without lower validation error. Parameter movement alone is insufficient; the tangent span and prediction must change usefully.